# Exploratory Colab experiment

> Cleaned archive of the original graduation-project notebook. For new leakage-aware runs, use the reusable pipeline under src/ and scripts/.


In [ ]:
# Gerekli kütüphaneler
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

# Sabitler
IMG_SIZE    = 299     # ResNet50 için giriş boyutu
BATCH_SIZE  = 32
INITIAL_LR  = 1e-4
EPOCHS_FT   = 80      # Fine-Tuning epoch sayısı

# Yollar
BASE_DIR   = 'data/split'
TRAIN_DIR  = os.path.join(BASE_DIR, 'train')
TEST_DIR   = os.path.join(BASE_DIR, 'test')
MODEL_PATH = 'artifacts/resnet50_eniyi.h5'
SAVE_PATH  = 'artifacts/resnet50_ft75_finetuned.h5'

# 1) Modeli yükle
model = load_model(MODEL_PATH)

# 2) Son 75 katmanı aç
for layer in model.layers[:-75]:
    layer.trainable = False
for layer in model.layers[-75:]:
    layer.trainable = True

# 3) Compile
model.compile(optimizer=Adam(learning_rate=INITIAL_LR * 0.1),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 4) Callback’ler
checkpoint = ModelCheckpoint(SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-7)

# 5) Data Augmentation & Generator’lar
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# 6) Class weights
y_train = train_gen.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# 7) Eğitim
history_ft = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FT,
    class_weight=class_weights,
    callbacks=[checkpoint, lr_reduce]
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_score, recall_score, f1_score
from sklearn.preprocessing import label_binarize
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

# === Parametreler ===
IMG_SIZE = 299
BATCH_SIZE = 32
MODEL_PATH = 'artifacts/resnet50_ft75_finetuned.h5'
TEST_DIR = 'data/split/test'

# === Test Generator ===
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# === Modeli Yükle ===
model = load_model(MODEL_PATH)

# === Test Seti Değerlendirmesi (Loss & Accuracy) ===
loss, acc = model.evaluate(test_gen, verbose=1)
print(f"\n🧪 Test Loss     : {loss:.4f}")
print(f"✅ Test Accuracy : {acc:.4f}")

# === Tahminler ===
Y_pred = model.predict(test_gen, verbose=1)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_gen.classes
labels = list(test_gen.class_indices.keys())
num_classes = len(labels)

# === Confusion Matrix ===
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# === Classification Report ===
print("📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=labels))

# === F1, Precision, Recall, Specificity ===
precision = precision_score(y_true, y_pred, average=None)
recall = recall_score(y_true, y_pred, average=None)
f1 = f1_score(y_true, y_pred, average=None)
specificity = []

for i in range(num_classes):
    tn = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
    fp = np.sum(np.delete(cm, i, axis=0)[:, i])
    specificity.append(tn / (tn + fp))

print("\n🎯 Class-wise Metrics:")
for i in range(num_classes):
    print(f"\nClass: {labels[i]}")
    print(f"Precision   : {precision[i]:.4f}")
    print(f"Recall      : {recall[i]:.4f}")
    print(f"Specificity : {specificity[i]:.4f}")
    print(f"F1 Score    : {f1[i]:.4f}")

# === ROC Eğrisi ===
y_true_bin = label_binarize(y_true, classes=range(num_classes))
fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], Y_pred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8, 6))
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label=f"Class {labels[i]} (AUC = {roc_auc[i]:.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.title("ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid()
plt.show()


In [ ]:
import pickle
from tensorflow.keras.models import load_model

# 1) MODELİ KAYDET (ağırlıklar + mimari birlikte)
MODEL_SAVE_PATH = 'artifacts/resnet50_final_with_history.h5'
model.save(MODEL_SAVE_PATH)
print(f"✅ Model başarıyla kaydedildi: {MODEL_SAVE_PATH}")

# 2) EĞİTİM GEÇMİŞİNİ KAYDET (accuracy, loss vs.)
HISTORY_SAVE_PATH = 'artifacts/resnet50_history.pkl'
with open(HISTORY_SAVE_PATH, 'wb') as f:
    pickle.dump(history_ft.history, f)
print(f"📈 Eğitim geçmişi kaydedildi: {HISTORY_SAVE_PATH}")
